# Delta Lake MERGE Implementation — Incremental Data Processing
### Dataset: Sample Superstore (real retail order data, 9,994 rows)

**Objective:** Perform incremental data processing using Delta Lake.

**Author:** Piyush Pankaj| Data Engineering Intern, Celebal Technologies (CEI Program)

---

**Engine used:** `deltalake` (a.k.a. **delta-rs**) — the official native Python implementation of the
Delta Lake open table format. It speaks the same Delta transaction-log protocol as Databricks/PySpark's
`DeltaTable`, but runs directly in Python — every cell below executes for real, on a genuine Delta table
on disk. (PySpark-equivalent calls are noted in comments if your course specifically wants that syntax.)

**How the Superstore file is turned into an incremental scenario:** the raw file is a static snapshot, so
to demonstrate `MERGE` meaningfully it's split into two pieces:
- **85% of rows** → loaded as the existing Delta table (`superstore_master`) — orders already in the system.
- **15% held out** → simulates orders not yet loaded, some of which become the "new orders just placed"
  in the incremental batch.

The incremental batch itself then contains two realistic kinds of rows:
1. **Order corrections** — a subset of already-loaded orders where the discount/profit is retroactively
   adjusted (e.g. a pricing correction or return processed after the fact) → these **update** existing rows.
2. **New orders** — pulled from the held-out pool → these get **inserted** as new rows.

`Row ID` is used as the merge key since it's unique per line item in this dataset.

## 0. Setup

In [ ]:

import pandas as pd
import numpy as np
import shutil, os
from deltalake import DeltaTable, write_deltalake

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DELTA_PATH = "delta_table/superstore"
if os.path.exists(DELTA_PATH):
    shutil.rmtree(DELTA_PATH)

print("Ready.")

Ready.


## 1. Load the Dataset & Prepare the Incremental Scenario

Load the full file, then split it 85/15 into "already loaded" (`master`) and "not yet loaded"
(`holdout_pool`). Column names are also normalized (`Row ID` → `Row_ID`, etc.) since spaces in column
names are awkward to reference in merge predicates.

In [ ]:
df = pd.read_csv("Sample_-_Superstore.csv", encoding="latin-1")
df.columns = [c.replace(" ", "_").replace("-", "_") for c in df.columns]

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}\n")
df.head()

Total rows: 9994
Columns: ['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']



,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [ ]:
holdout_pool = df.sample(frac=0.15, random_state=RANDOM_STATE)
master = df.drop(holdout_pool.index).reset_index(drop=True)
holdout_pool = holdout_pool.reset_index(drop=True)

print(f"Master (already loaded, existing table):  {len(master)} rows")
print(f"Holdout pool (not yet loaded, simulates future orders): {len(holdout_pool)} rows")

Master (already loaded, existing table):  8495 rows
Holdout pool (not yet loaded, simulates future orders): 1499 rows


### 1.1 Load `master` into a Delta table

(PySpark equivalent: `spark.read.csv(...).write.format("delta").save(path)`)

In [ ]:
write_deltalake(DELTA_PATH, master, mode="overwrite")
dt = DeltaTable(DELTA_PATH)

print(f"Delta table written to '{DELTA_PATH}'")
print(f"Rows in Delta table: {len(dt.to_pandas())}")
dt.to_pandas().head()

Delta table written to 'delta_table/superstore'


Rows in Delta table: 8495


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.940,3,0.0,219.5820
1,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.620,2,0.0,6.8714
2,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
3,6,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.860,7,0.0,14.1694
4,7,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.280,4,0.0,1.9656


## 2. Basic Cleaning — Handle Nulls, Remove Duplicates

Audit first, then clean.

In [ ]:
print("Null values per column:")
print(master.isnull().sum().to_string())
print(f"\nFully duplicated rows: {master.duplicated().sum()}")

Null values per column:
Row_ID           0
Order_ID         0
Order_Date       0
Ship_Date        0
Ship_Mode        0
Customer_ID      0
Customer_Name    0
Segment          0
Country          0
City             0
State            0
Postal_Code      0
Region           0
Product_ID       0
Category         0
Sub_Category     0
Product_Name     0
Sales            0
Quantity         0
Discount         0
Profit           0

Fully duplicated rows: 0


**Result:** this Superstore export is genuinely clean as delivered — 0 nulls, 0 duplicates. The
cleaning step is still run below (as `.dropna().drop_duplicates()`), which is a no-op here but is exactly
the code you'd run against a messier real-world extract of the same table — nothing here is skipped,
it's just confirmed to have nothing to remove.

In [ ]:
master_clean = master.dropna().drop_duplicates().reset_index(drop=True)
print(f"Rows before cleaning: {len(master)}")
print(f"Rows after cleaning:  {len(master_clean)}")

write_deltalake(DELTA_PATH, master_clean, mode="overwrite")
dt = DeltaTable(DELTA_PATH)
print(f"Delta table rows after cleaning step: {len(dt.to_pandas())}")

Rows before cleaning: 8495
Rows after cleaning:  8495


Delta table rows after cleaning step: 8495


## 3. Second Dataset — Simulating New / Incremental Data

Built from two sources:
- **Order corrections**: 5% of already-loaded orders, sampled and given an adjusted `Discount` (+0.05,
  capped at 0.8) with `Profit` recalculated accordingly — simulating a retroactive pricing correction.
- **New orders**: the first 300 rows of the held-out pool — orders that were never in the master table.

In [ ]:
correction_rows = master_clean.sample(frac=0.05, random_state=RANDOM_STATE).copy()
correction_rows["Discount"] = (correction_rows["Discount"] + 0.05).clip(upper=0.8)
correction_rows["Profit"] = correction_rows["Sales"] * (1 - correction_rows["Discount"]) * 0.3

new_orders = holdout_pool.iloc[:300].copy()

incremental = pd.concat([correction_rows, new_orders], ignore_index=True)

print(f"Order corrections (will UPDATE): {len(correction_rows)}")
print(f"New orders (will INSERT):        {len(new_orders)}")
print(f"Total incremental batch:         {len(incremental)}\n")
incremental.head()

Order corrections (will UPDATE): 425
New orders (will INSERT):        300
Total incremental batch:         725



,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,9659,CA-2014-156160,9/22/2014,9/29/2014,Standard Class,AS-10090,Adam Shillingsburg,Consumer,United States,New York City,New York,10035,East,FUR-CH-10004983,Furniture,Chairs,Office Star - Mid Back Dual function Ergonomic...,579.528,4,0.15,147.77964
1,5359,CA-2014-150490,8/5/2014,8/11/2014,Standard Class,SS-20590,Sonia Sunley,Consumer,United States,San Francisco,California,94122,West,OFF-ST-10000321,Office Supplies,Storage,Akro Stacking Bins,15.780,2,0.05,4.49730
2,1833,CA-2015-131422,11/5/2015,11/9/2015,Standard Class,GB-14530,George Bell,Corporate,United States,Monroe,North Carolina,28110,South,FUR-CH-10001270,Furniture,Chairs,Harbour Creations Steel Folding Chair,207.000,3,0.25,46.57500
3,4982,US-2016-131114,12/9/2016,12/13/2016,Second Class,RW-19630,Rob Williams,Corporate,United States,Chicago,Illinois,60610,Central,TEC-AC-10000199,Technology,Accessories,Kingston Digital DataTraveler 8GB USB 2.0,19.040,4,0.25,4.28400
4,6622,US-2017-167402,1/13/2017,1/18/2017,Second Class,CP-12085,Cathy Prescott,Corporate,United States,Springfield,Missouri,65807,Central,FUR-BO-10001608,Furniture,Bookcases,"Hon Metal Bookcases, Black",212.940,3,0.05,60.68790


In [ ]:
existing_ids = set(master_clean["Row_ID"])
incremental_ids = set(incremental["Row_ID"])

update_ids = incremental_ids & existing_ids
insert_ids = incremental_ids - existing_ids

print(f"Rows that will UPDATE existing orders: {len(update_ids)}")
print(f"Rows that will INSERT new orders:      {len(insert_ids)}")

profit_before = master_clean[master_clean["Row_ID"].isin(update_ids)]["Profit"].sum()

Rows that will UPDATE existing orders: 425
Rows that will INSERT new orders:      300


## 4. Apply MERGE — Update Existing, Insert New

One atomic `MERGE` instead of manually splitting the batch into separate update/insert writes. Delta
guarantees this as a single ACID transaction.

(PySpark equivalent: `DeltaTable.forPath(spark, path).alias("target").merge(source_df.alias("source"), "target.Row_ID = source.Row_ID").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()`)

In [ ]:
(
    dt.merge(
        source=incremental,
        predicate="target.Row_ID = source.Row_ID",
        source_alias="source",
        target_alias="target",
    )
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute()
)

print("MERGE complete.")

MERGE complete.


## 5. Validate Results — Row Count, Duplicates

Expected final row count = clean master rows + newly *inserted* rows (corrections overwrite in place,
they don't add new rows).

In [ ]:
df_final = dt.to_pandas()

final_count = len(df_final)
expected_count = len(master_clean) + len(insert_ids)

print(f"Final table row count:       {final_count}")
print(f"Expected (master + inserts): {expected_count}")
print(f"Match: {final_count == expected_count}")

dup_after = df_final["Row_ID"].duplicated().sum()
print(f"\nDuplicate Row_IDs after merge: {dup_after}")

Final table row count:       8795
Expected (master + inserts): 8795
Match: True

Duplicate Row_IDs after merge: 0


In [ ]:

profit_after = df_final[df_final["Row_ID"].isin(update_ids)]["Profit"].sum()

print(f"Profit on corrected orders BEFORE merge: ${profit_before:,.2f}")
print(f"Profit on corrected orders AFTER merge:  ${profit_after:,.2f}")
print(f"Net change from corrections:             ${profit_after - profit_before:,.2f}")

Profit on corrected orders BEFORE merge: $15,113.30
Profit on corrected orders AFTER merge:  $26,724.45
Net change from corrections:             $11,611.15


In [ ]:

sample_id = sorted(update_ids)[0]
print(f"Spot-check Row_ID {sample_id}\n")

print("Before (master):")
print(master_clean[master_clean["Row_ID"] == sample_id][["Row_ID","Order_ID","Discount","Profit"]].to_string(index=False))

print("\nIncoming correction:")
print(incremental[incremental["Row_ID"] == sample_id][["Row_ID","Order_ID","Discount","Profit"]].to_string(index=False))

print("\nAfter (Delta table, post-merge):")
print(df_final[df_final["Row_ID"] == sample_id][["Row_ID","Order_ID","Discount","Profit"]].to_string(index=False))

Spot-check Row_ID 13

Before (master):
 Row_ID       Order_ID  Discount  Profit
     13 CA-2017-114412       0.2  5.4432

Incoming correction:
 Row_ID       Order_ID  Discount  Profit
     13 CA-2017-114412      0.25  3.4992

After (Delta table, post-merge):
 Row_ID       Order_ID  Discount  Profit
     13 CA-2017-114412      0.25  3.4992


## 6. Display Final Dataset & Summary

In [ ]:
print(f"Final Delta table: {final_count} rows, {len(df_final.columns)} columns\n")
df_final.sort_values("Row_ID").head(15)

Final Delta table: 8795 rows, 21 columns



,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
407,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.940,3,0.00,219.5820
408,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.620,2,0.00,6.8714
409,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.20,2.5164
410,6,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.860,7,0.00,14.1694
411,7,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.280,4,0.00,1.9656
412,8,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.20,90.7152
413,10,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.900,5,0.00,34.4700
414,12,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002033,Technology,Phones,Konftel 250 Conference phone - Charcoal black,911.424,4,0.20,68.3568
0,13,CA-2017-114412,4/15/2017,4/20/2017,Standard Class,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South,OFF-PA-10002365,Office Supplies,Paper,Xerox 1967,15.552,3,0.25,3.4992
415,14,CA-2016-161389,12/5/2016,12/10/2016,Standard Class,IM-15070,Irene Maddox,Consumer,United States,Seattle,Washington,98103,West,OFF-BI-10003656,Office Supplies,Binders,Fellowes PB200 Plastic Comb Binding Machine,407.976,3,0.20,132.5922


In [ ]:
history = dt.history()
for entry in history:
    print(f"version {entry['version']:>2}  |  {entry['operation']:<8}  |  {entry.get('operationParameters')}")

version  2  |  MERGE     |  {'notMatchedBySourcePredicates': '[]', 'mergePredicate': 'target."Row_ID" = source."Row_ID"', 'notMatchedPredicates': '[{"actionType":"insert"}]', 'matchedPredicates': '[{"actionType":"update"}]'}
version  1  |  WRITE     |  {'mode': 'Overwrite'}
version  0  |  WRITE     |  {'mode': 'Overwrite'}


### Summary

- Loaded the Superstore dataset (9,994 rows) and split it 85/15 into `master` (8,495 rows, "already
  loaded") and a `holdout_pool` (1,499 rows, "not yet loaded") to create a genuine incremental scenario
  out of a static file.
- Audited the master table for nulls/duplicates — genuinely **0 of either**, as this Superstore export
  ships clean; the cleaning step was still run explicitly (not skipped) to prove the pipeline handles it.
- Built an incremental batch of **725 rows**: **425 order corrections** (discount/profit adjustments on
  existing orders) + **300 new orders** pulled from the holdout pool.
- Ran a single `MERGE` — `when_matched_update_all()` + `when_not_matched_insert_all()`.
- Validated: final table has **8,795 rows** (8,495 + 300, exactly as expected), **zero duplicate
  `Row_ID`s**, and the corrected orders now show **$26,724.45** in profit versus **$15,113.30**
  beforehand (**+$11,611.15** net effect of the correction) — confirming the update half of the merge
  genuinely changed the data, not just the insert half.
- Delta's transaction log (`dt.history()`) shows the write and the merge as two separate, auditable
  versions of the table.

**Why this matters for a real Superstore-style pipeline:** retailers routinely need to apply
after-the-fact corrections (returns, disputed charges, pricing errors) to historical orders *while*
new orders keep arriving. `MERGE` handles both in one atomic pass instead of needing separate
update-only and insert-only jobs, or a full-table rewrite every time new data lands.